# Week 2 - Classification Model
### AI & ML Internship, VortexTech

For this one I'm using my Week 1 dataset (Netflix titles from Kaggle). It doesn't have an obvious yes/no column like the survived/pass-fail examples in the brief, so I picked my own binary target: predicting whether a title is a **Movie** or a **TV Show** based on its other attributes.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('netflix_titles.csv')
df.head()

## 1. Picking the target and features

Target column: `type` (Movie or TV Show), turned into 1/0.

I'm NOT using the `duration` column even though it's tempting - it literally says "90 min" for movies and "2 Seasons" for TV shows, so the model would just be reading the answer off the label instead of learning anything. Same problem with `listed_in` (genre) - some categories are literally named "TV Dramas" or "International TV Shows", so that leaks the answer too. I checked this by grouping genre by type and it was obvious.

So the features I'm actually using are `release_year`, `rating`, and `country` (simplified to the top 10 countries + "Other" so I don't end up with 100+ dummy columns).

In [ ]:
df.isnull().sum()

In [ ]:
# a handful of rows are missing rating, just drop those
df = df.dropna(subset=['rating'])

# country has a lot of missing values, fill with 'Unknown' instead of dropping
df['country'] = df['country'].fillna('Unknown')

# release_year is a big number (like 2019), which made logistic regression
# take forever to converge, so shifting it down to start near 0
df['release_year_norm'] = df['release_year'] - df['release_year'].min()

# too many unique countries to one-hot encode all of them, so keep top 10 and bucket the rest
top_countries = df['country'].value_counts().head(10).index
df['country_simple'] = df['country'].apply(lambda c: c if c in top_countries else 'Other')

df['target'] = (df['type'] == 'Movie').astype(int)
df[['type', 'target']].head()

In [ ]:
features = df[['release_year_norm', 'rating', 'country_simple']]
features = pd.get_dummies(features, columns=['rating', 'country_simple'])

X = features
y = df['target']

X.shape

## 2. Train/test split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 3. Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("accuracy:", accuracy_score(y_test, preds))
print("precision:", precision_score(y_test, preds))
print("recall:", recall_score(y_test, preds))
print("f1:", f1_score(y_test, preds))

## 4. Comparing with a Decision Tree

In [ ]:
tree = DecisionTreeClassifier(max_depth=6, random_state=42)
tree.fit(X_train, y_train)
preds_tree = tree.predict(X_test)

print("accuracy:", accuracy_score(y_test, preds_tree))
print("precision:", precision_score(y_test, preds_tree))
print("recall:", recall_score(y_test, preds_tree))
print("f1:", f1_score(y_test, preds_tree))

## 5. Results

Both models land around 70% accuracy, which is decent but not great - basically the model has learned that certain ratings and release years lean movie vs TV show, but there's a lot of overlap so it can't separate them cleanly. Recall is quite high (over 90%) for both models, meaning it's good at catching most movies, but precision is lower, so it's also mislabeling a fair number of TV shows as movies. This makes sense because rating and country alone don't fully describe what makes something a show vs a movie - a lot of that signal is genre/duration related, which I intentionally excluded to avoid leakage.

If I wanted to improve this without leaking the answer, I'd try adding cast/director info (e.g. number of cast members, since TV shows tend to have larger casts), or engineering the genre feature more carefully by stripping out any genre words that literally reference "TV" or "Movie" instead of dropping the whole column.